This Python script connects to the Ethereum Mainnet via Alchamy and extracts deployed smart contract bytecode from a given block. It retrieves all transactions, filters out those interacting with smart contracts, converts addresses to checksum format, and fetches the corresponding bytecode. The extracted bytecode is then saved in a text file for further analysis, making it useful for blockchain researchers, developers, and security analysts.

In [1]:
!pip install web3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.5/587.5 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.5/102.5 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 63.6 MB/s eta 0:00:00


In [2]:
import os
import json
import random
import shutil
from web3 import Web3

In [3]:
# Set up Web3 connection (Replace with your Infura/Alchemy endpoint)
INFURA_URL = "https://eth-mainnet.g.alchemy.com/v2/HS3F7LNSY5Gex3PjW5xHfaelqbwrLyxm"
web3 = Web3(Web3.HTTPProvider(INFURA_URL))

In [4]:
def to_checksum(addr):
    """Convert address to checksum format."""
    return Web3.to_checksum_address(addr)

def get_contract_transactions(block_number):
    """Fetch transactions where the recipient is a smart contract."""
    block = web3.eth.get_block(block_number, full_transactions=True)
    contract_txs = {}

    for tx in block.transactions:
        if tx.to and web3.eth.get_code(to_checksum(tx.to)) != b"":  # Ensure recipient is a contract
            contract_address = to_checksum(tx.to)
            if contract_address not in contract_txs:
                contract_txs[contract_address] = []
            contract_txs[contract_address].append(tx)

    return contract_txs  # Dictionary {contract_address: [tx1, tx2, ...]}

def get_contract_bytecode(contract_address, block_number):
    """Fetch deployed bytecode for the contract at a specific block."""
    bytecode = web3.eth.get_code(to_checksum(contract_address), block_identifier=block_number)
    return bytecode.hex() if bytecode else None

def save_bytecode(contract_address, bytecode, folderName):
    """Save contract bytecode to a file."""
    os.makedirs(folderName, exist_ok=True)
    file_path = os.path.join(folderName, f"{contract_address}.txt")
    with open(file_path, "w") as f:
        f.write(bytecode)

def get_transaction_input_data(transactions):
    """Fetch input data from a transaction that interacted with the contract."""

    selected_tx = transactions[0]
    #print(f"herrrrrrrrrrr {selected_tx}")
    tx_hash = selected_tx.hash.hex()
    tx = web3.eth.get_transaction(tx_hash)

    input_data = tx.input.hex() if tx.input else None  # Convert HexBytes to hex string

    if input_data:  # Found valid input data
        return tx_hash, format_input_data(input_data)

    return tx_hash, "0" * 256  # No valid transaction found

def format_input_data(input_data):
    """Ensure input data is exactly 720 characters (90 bytes)."""
    lenInHex = 180
    if len(input_data) < lenInHex:
        return input_data.ljust(lenInHex, '0')  # Pad with zeros
    return input_data[:lenInHex]  # Truncate if too long

def main(block_number, folderName):
    """Main function to fetch contract bytecode and input data from a block."""
    if not web3.is_connected():
        print("❌ Failed to connect to Ethereum node.")
        return

    print(f"🔍 Fetching contract transactions from block {block_number}...")
    contract_txs = get_contract_transactions(block_number)  # Get contracts & their transactions

    if not contract_txs:
        print("⚠️ No contract transactions found.")
        return

    print(f"✅ Found {len(contract_txs)} contract addresses.")

    for contract, transactions in contract_txs.items():
        # Save bytecode
        bytecode = get_contract_bytecode(contract, block_number)
        if bytecode:
            save_bytecode(contract, bytecode, folderName)
        else:
            print(f"⚠️ No bytecode found for {contract}")

        # Save input data from a contract-related transaction
        tx_hash, input_data = get_transaction_input_data(transactions)
        if tx_hash and input_data:
            folderNameCallData = "inputData.txt";
            calDataChunks = 12;
            chunks = chunk_hex_data(input_data, calDataChunks)
            for chunk in chunks:
                save_input_data(folderName, chunk, folderNameCallData)
            folderNameCallData = "inputDataOrig.txt";
            save_input_data(folderName, input_data, folderNameCallData)
            #print(tx_hash)
        else:
            print(f"⚠️ No valid transaction with input data found for {contract}")

def chunk_hex_data(data: str, n: int):
    """
    Split hex data into chunks of size n, padding the last chunk with zeros if needed.

    Args:
        data (str): Hexadecimal string (without '0x').
        n (int): Chunk size in characters.

    Returns:
        List[str]: List of equal-length chunks.
    """
    if n <= 0:
        raise ValueError("Chunk size must be a positive integer")

    # Pad data with zeros to make length a multiple of n
    remainder = len(data) % n
    if remainder != 0:
        padding = n - remainder
        data += '0' * padding

    return [data[i:i + n] for i in range(0, len(data), n)]


def save_input_data(folderName, input_data, folderNameCallData):
    """Save input data to a file inside the specified folder."""
    os.makedirs(folderName, exist_ok=True)
    file_path = os.path.join(folderName, folderNameCallData)

    with open(file_path, "a") as f:  # Append to a single file
        f.write(f"{input_data}\n")

def format_hex_file(input_file, output_file, n):
    with open(input_file, 'r') as infile:
        data = infile.read().strip()

    # Pad the data with zeros if necessary
    if len(data) % n != 0:
        data = data.ljust(len(data) + (n - len(data) % n), '0')

    # Split data into chunks of size n
    chunks = [data[i:i + n] for i in range(0, len(data), n)]

    # Write the chunks to the output file
    with open(output_file, 'w') as outfile:
        outfile.write('\n'.join(chunks))

def copy_txt_file(src_folder, dest_folder, filename):
    """Copy a text file from the source folder to the destination folder."""
    os.makedirs(dest_folder, exist_ok=True)  # Ensure destination folder exists

    src_path = os.path.join(src_folder, filename)
    dest_path = os.path.join(dest_folder, filename)

    if os.path.exists(src_path):  # Check if the source file exists
        shutil.copy(src_path, dest_path)  # Copy file
        print(f"✅ {filename} copied to {dest_folder}")
    else:
        print(f"⚠️ {filename} not found in {src_folder}")


In [5]:
ethBlocks = [ #6653186,
              #6653197,
              #6653232,
              6653208,
              #6653220,
              #6653205,
              #6653209
              ]

for block_number in ethBlocks:
  folderName = "Block"+str(block_number)
  main(block_number, folderName)
   # Up to here we get data raw data from Etherscan and stor in block folder

🔍 Fetching contract transactions from block 6653208...
✅ Found 32 contract addresses.


In [6]:
# Create the EVM folder
# Chunk each bytecode file and store them in EVM foler
# Copy inputData.txt to EVM folder

for block_number in ethBlocks:
  folderName = "Block"+str(block_number)
  n = 64 # Len in hex of opcodes to load to BYTECODE_MEM at the same time

   # Process data for EVM
  input_folder = folderName # The newly created folders
  output_folder = folderName + "_EVM"

  # Ensure output directory exists
  os.makedirs(output_folder, exist_ok=True)

  # Process each .txt file in the input folder
  for filename in os.listdir(input_folder):
    if filename == "inputDataOrig.txt":
      print("inputDataOrig.txt skiped")
    elif filename == "inputData.txt":
        # Example usage:
        copy_txt_file(input_folder, output_folder, "inputData.txt")

    elif filename.endswith(".txt"):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, filename)

        # Read the input file
        with open(input_path, "r") as file:
            data = file.read().strip()  # Read and remove extra spaces/newlines

        # Pad the data with zeros if necessary
        if len(data) % n != 0:
            data = data.ljust(len(data) + (n - len(data) % n), '0')

        # Split data into chunks of size n
        chunks = [data[i:i + n] for i in range(0, len(data), n)]

        # Write the chunks to the output file
        with open(output_path, 'w') as file:
            file.write('\n'.join(chunks))


✅ inputData.txt copied to Block6653208_EVM
inputDataOrig.txt skiped


In [7]:
  # Create file_list containing directories to each txt file with bytecode
  # Create a list of file names
  import os
  #C:\Users\AR43170\Desktop\PhD_desktop\PHD_Codes\Gitlab\poncha\vhdl\1_PhD\EVM\ETH_BLOCKS\Block6653186_EVM\0x0ABeFb7611Cb3A01EA3FaD85f33C3C934F8e2cF4.txt
  # Folder containing the txt files



for block_number in ethBlocks:
    folderName = "Block"+str(block_number)
    output_folder = folderName + "_EVM"

    folder_path = "/content/" + output_folder
    outputFolder = output_folder
    output_file = os.path.join(outputFolder, "file_list.txt")

    try:
        # Ensure the output folder exists
        os.makedirs(outputFolder, exist_ok=True)

        # Get all txt files in the folder
        txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

        # Write file names with "myFile " prefix
        with open(output_file, "w") as f:
            for file_name in txt_files:
              if file_name != "file_list.txt" and file_name != "inputData.txt":
                f.write(f"C:\\Users\\AR43170\\Desktop\\PhD_desktop\\PHD_Codes\\Gitlab\\poncha\\vhdl\\1_PhD\\EVM\\ETH_BLOCKS_IMP\\{output_folder}\\{file_name}\n") # PC
                #f.write(f"C:\\Users\\ponch\\Desktop\\myPhDCodes\\Gitlab\\poncha\\vhdl\\1_PhD\\EVM\\ETH_BLOCKS_IMP\\{output_folder}\\{file_name}\n") # Laptop
        print(f"File names saved in {output_file}")

    except Exception as e:
        print(f"Error: {e}")

File names saved in Block6653208_EVM/file_list.txt


In [8]:
# Convert to zip foles to download
!zip -r /content/Block6653186_EVM.zip /content/Block6653186_EVM
!zip -r /content/Block6653197_EVM.zip /content/Block6653197_EVM
!zip -r /content/Block6653205_EVM.zip /content/Block6653205_EVM
!zip -r /content/Block6653208_EVM.zip /content/Block6653208_EVM
!zip -r /content/Block6653209_EVM.zip /content/Block6653209_EVM
!zip -r /content/Block6653220_EVM.zip /content/Block6653220_EVM
!zip -r /content/Block6653232_EVM.zip /content/Block6653232_EVM

	zip warning: name not matched: /content/Block6653186_EVM

zip error: Nothing to do! (try: zip -r /content/Block6653186_EVM.zip . -i /content/Block6653186_EVM)
	zip warning: name not matched: /content/Block6653197_EVM

zip error: Nothing to do! (try: zip -r /content/Block6653197_EVM.zip . -i /content/Block6653197_EVM)
	zip warning: name not matched: /content/Block6653205_EVM

zip error: Nothing to do! (try: zip -r /content/Block6653205_EVM.zip . -i /content/Block6653205_EVM)
  adding: content/Block6653208_EVM/ (stored 0%)
  adding: content/Block6653208_EVM/0x818E6FECD516Ecc3849DAf6845e3EC868087B755.txt (deflated 73%)
  adding: content/Block6653208_EVM/file_list.txt (deflated 80%)
  adding: content/Block6653208_EVM/0x7003C6a38cAd0b76c52A059e082440c946F8769D.txt (deflated 70%)
  adding: content/Block6653208_EVM/0x1A753721ECAe08490F7505aB30BfbB6BeCa6224b.txt (deflated 85%)
  adding: content/Block6653208_EVM/0xb89570f6AD742CB1fd440A930D6c2A2eA29c51eE.txt (deflated 77%)
  adding: content/Bl

File names saved in Block6653209_EVM/file_list.txt
